# Regressione logistica

In [1]:
import pandas as pd
import numpy as np
import re
import math
import time
from pathlib import Path
import warnings
from datetime import timedelta
from tabulate import tabulate
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer, roc_auc_score, accuracy_score
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [2]:
def training(file_path, csv_name):

    # Lettura dati
    df = pd.read_csv(file_path)

    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]', 'HER2 [SII]']

    # Rimuovo righe senza target validi
    df_validi = df.dropna(subset=original_target_list).copy()

    # Binarizzazione target
    df_validi['PR_class']   = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class']   = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    df_validi['HER2_class'] = (df_validi['HER2 [SII]'] >= 3).astype(int)

    final_target_list = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign',
                        'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    # Per coerenza con gli altri modelli: 4 fold per paziente
    cv = GroupKFold(n_splits=4)

    # Pipeline: StandardScaler + LogisticRegression (L2 fissa)
    logistic_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            random_state=42,
            n_jobs=1,
            class_weight='balanced',
            solver='saga',
            penalty='l2',
            max_iter=2000,
            tol=1e-4
        ))
    ])

    multi_output_model = MultiOutputClassifier(logistic_pipeline)


    iperparametri = {
        'estimator__classifier__C': [0.1, 1.0]
    }


    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i],
                                   average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    total_combinations = len(iperparametri['estimator__classifier__C'])
    print(f"\nInizio Grid Search Logistic (GRID MINIMAL: {total_combinations} combinazioni) per: {csv_name}")

    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', 'overflow encountered')
        warnings.filterwarnings('ignore', 'invalid value encountered')
        warnings.filterwarnings('ignore', 'ConvergenceWarning')

        grid_search = GridSearchCV(
            estimator=multi_output_model,
            param_grid=iperparametri,
            cv=cv,
            scoring=scorer,
            n_jobs=-1,
            verbose=1,
            refit=True,
            error_score=0.0
        )

        grid_search.fit(features, target, groups=groups)

    # --- RECUPERO RISULTATI ---
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_

    fold_reports = []

    # Pulisco i nomi dei parametri (solo C) e aggiungo penalty='l2' fisso
    clean_param_dict = {
        'C': best_params['estimator__classifier__C'],
        'penalty': 'l2'
    }

    # Metriche per FOLD e per LABEL
    fold_reports = []

    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

        # Pipeline fresca per ogni fold
        fresh_pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(
                random_state=42,
                n_jobs=1,
                class_weight='balanced',
                solver='saga',
                max_iter=2000,
                tol=1e-4,
                penalty='l2',
                C=clean_param_dict['C']   # iperparametro ottimizzato
            ))
        ])

        model_clone = MultiOutputClassifier(fresh_pipeline)
        model_clone.fit(X_train, y_train)

        y_pred = model_clone.predict(X_test)
        y_proba_list = model_clone.predict_proba(X_test)

        fold_metrics = {}
        for i, col in enumerate(final_target_list):
            y_true_i = y_test.iloc[:, i]
            y_pred_i = y_pred[:, i]

            # F1 e Accuracy
            f1 = f1_score(y_true_i, y_pred_i, zero_division=0)
            acc = accuracy_score(y_true_i, y_pred_i)

            # AUC
            unique_classes = np.unique(y_true_i)
            if len(unique_classes) < 2:
                auc_val = np.nan
            else:
                try:
                    # per LR binaria mi aspetto 2 colonne: p(classe 0), p(classe 1)
                    if y_proba_list[i].shape[1] == 2:
                        auc_val = roc_auc_score(y_true_i, y_proba_list[i][:, 1])
                    else:
                        # caso degno di nota ma raro (multi-classe)
                        auc_val = 0.5
                except ValueError:
                    auc_val = np.nan

            fold_metrics[col] = {
                'f1': f1,
                'accuracy': acc,
                'auc': auc_val
            }

        fold_reports.append(fold_metrics)

    final_result = [{
        **clean_param_dict,  # qui ci metti i parametri ottimali della LR (es. C)
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result


# Stampo i risultati in un formato leggibile

In [3]:
def print_grid_search_results(results_per_dataset):

    print("\n" + "=" * 80)
    print(" " * 20 + "Migliori metrice per ogni fold")
    print("=" * 80)

    for dataset_name, metrics_list in results_per_dataset.items():
        if not metrics_list:
            continue

        best_result = metrics_list[0]     # modello migliore per il dataset
        fold_reports = best_result["fold_reports"]


        print(f"\n\nDataset: {dataset_name}")
        print("-" * 80)

        target_names = ['PR_class', 'ER_class', 'KI67_class', 'HER2_class']

        for target in target_names:

            # --- LISTE DI VALORI SUI FOLD ---
            f1_list  = np.array([fold[target]["f1"] for fold in fold_reports])
            acc_list = np.array([fold[target]["accuracy"] for fold in fold_reports])
            auc_list = np.array([fold[target]["auc"] for fold in fold_reports], dtype=float)

            # --- BEST VALUES ---
            best_f1  = np.max(f1_list)
            best_acc = np.max(acc_list)

            # Per AUC rimuovo eventuali NaN
            valid_auc = ~np.isnan(auc_list)
            best_auc  = np.max(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STD DEV ---
            f1_std  = np.std(f1_list)
            acc_std = np.std(acc_list)
            auc_std = np.std(auc_list[valid_auc]) if valid_auc.any() else np.nan

            # --- STAMPO RISULTATI ---
            print(f"\nTarget: {target}")
            print(f"  F1-score     = {best_f1:.3f} " +" ± "+f" {f1_std:.3f}")
            print(f"  Accuracy     = {best_acc:.3f} " +" ± "+f" {acc_std:.3f}")
            print(f"  AUC          = {best_auc:.3f} " +" ± "+f" {auc_std:.3f}" if not np.isnan(best_auc) else
                  f"  AUC          = NaN           " +" ± "+f" NaN")


# Eseguo il tutto

In [4]:
start_time = time.time()


# Eseguo il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)



end_time = time.time()
# Calcolo il tempo impiegato
execution_time = end_time - start_time
formatted_time = str(timedelta(seconds=int(execution_time)))

print("\n" + "=" * 80)
print(f" TEMPO TOTALE DI ESECUZIONE: {formatted_time}")
print("=" * 80 + "\n")



Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: t2_medsam
Fitting 4 folds for each of 2 candidates, totalling 8 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: t2_preprocessed
Fitting 4 folds for each of 2 candidates, totalling 8 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: t2_original
Fitting 4 folds for each of 2 candidates, totalling 8 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-


Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: medsam_dynamic
Fitting 4 folds for each of 2 candidates, totalling 8 fits

Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: preprocessed_dynamic
Fitting 4 folds for each of 2 candidates, totalling 8 fits


/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/francesco/Tesi/BC-ML4/myenv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/fra


Inizio Grid Search Logistic (GRID MINIMAL: 2 combinazioni) per: original_dynamic
Fitting 4 folds for each of 2 candidates, totalling 8 fits

                    Migliori metrice per ogni fold


Dataset: t2_medsam
--------------------------------------------------------------------------------

Target: PR_class
  F1-score     = 0.947  ±  0.154
  Accuracy     = 0.933  ±  0.156
  AUC          = 0.889  ±  0.151

Target: ER_class
  F1-score     = 0.800  ±  0.063
  Accuracy     = 0.667  ±  0.072
  AUC          = 0.575  ±  0.170

Target: KI67_class
  F1-score     = 0.632  ±  0.102
  Accuracy     = 0.533  ±  0.122
  AUC          = 0.481  ±  0.119

Target: HER2_class
  F1-score     = 0.500  ±  0.217
  Accuracy     = 1.000  ±  0.152
  AUC          = 0.857  ±  0.181


Dataset: t2_preprocessed
--------------------------------------------------------------------------------

Target: PR_class
  F1-score     = 0.889  ±  0.159
  Accuracy     = 0.867  ±  0.144
  AUC          = 0.870  ±  0.183

Target: